# Week 5 Assignment: Personal Knowledge Worker

This notebook implements an end-to-end conversational RAG workflow over a personal knowledge base, including document ingestion, chunking, vectorization, retrieval, reranking, QA, and Gradio UI.

---

## 1. Import Required Libraries and Dependencies
Import all necessary Python packages for the project.

In [8]:
# Imports
import os
import glob
import gradio as gr
import chromadb
from chromadb.config import Settings
from openai import OpenAI
from dotenv import load_dotenv

# Load environment variables
load_dotenv(override=True)
OPENAI_API_KEY = os.getenv('OPENROUTER_API_KEY')
BASE_URL="https://openrouter.ai/api/v1"
assert OPENAI_API_KEY, "OpenAI API key not found in environment variables!"


openai_client = OpenAI(api_key=OPENAI_API_KEY, base_url=BASE_URL)



---

## 2. Document Ingestion Utility
Function to ingest .md and .txt files from local folders, supporting typed subfolders.

In [9]:
# Document Ingestion Utility

def ingest_documents(folder_path):
    docs = []
    for root, dirs, files in os.walk(folder_path):
        for file in files:
            if file.endswith('.md') or file.endswith('.txt'):
                file_type = os.path.basename(root)
                with open(os.path.join(root, file), 'r', encoding='utf-8') as f:
                    docs.append({
                        'type': file_type,
                        'filename': file,
                        'text': f.read()
                    })
    return docs

# Example usage:
# docs = ingest_documents('knowledge_base/')

---

## 3. LLM-Based Intelligent Chunking Utility
Function for chunking documents using structured outputs (Chunk, Chunks) with headline, summary, and original text.

In [10]:
# LLM-Based Intelligent Chunking Utility

def chunk_document(doc_text, model="gpt-4.1-mini"):
    prompt = f"Chunk the following document into structured outputs. For each chunk, provide a headline, summary, and original text. Respond as a JSON array of objects with keys: headline, summary, text.\n\n{doc_text}"
    response = openai_client.chat.completions.create(model=model, messages=[{"role": "user", "content": prompt}])
    chunks = response.choices[0].message.content
    try:
        return eval(chunks)  # Use json.loads in production
    except Exception:
        return []


---

## 4. ChromaDB Vector Store Creation Utility
Function to create a ChromaDB collection with OpenAI embeddings and rebuild flow.

In [11]:
# ChromaDB Vector Store Creation Utility

def create_chromadb_collection(chunks, collection_name="personal_knowledge"):
    client = chromadb.Client(Settings())
    try:
        client.delete_collection(collection_name)
    except Exception:
        pass
    collection = client.create_collection(collection_name)
    for idx, chunk in enumerate(chunks):
        collection.add(
            documents=[chunk['text']],
            metadatas=[{'headline': chunk['headline'], 'summary': chunk['summary']}],
            ids=[str(idx)]
        )
    return collection

# Example usage:
# collection = create_chromadb_collection(chunks)

---

## 5. Advanced Retrieval Pipeline Utility
Functions for query rewriting (chat-aware), semantic retrieval, and LLM reranking.

In [12]:
# Advanced Retrieval Pipeline Utility

def rewrite_query(query, history, model="gpt-4.1-mini"):
    prompt = f"Rewrite the following query for retrieval, considering the conversation history: {history}\nQuery: {query}"
    response = openai_client.chat.completions.create(model=model, messages=[{"role": "user", "content": prompt}])
    return response.choices[0].message.content

def semantic_retrieve(collection, query, top_k=5):
    results = collection.query(query_texts=[query], n_results=top_k)
    return results

def rerank_results(results, query, model="gpt-4.1-mini"):
    docs = [doc for doc in results['documents'][0]]
    prompt = f"Rerank the following documents for the query: '{query}'. Respond with the best document.\nDocuments: {docs}"
    response = openai_client.chat.completions.create(model=model, messages=[{"role": "user", "content": prompt}])
    return response.choices[0].message.content

---

## 6. Conversational QA Utility
Function for source-aware context assembly and history-aware response generation.

In [13]:
# Conversational QA Utility

def conversational_qa(query, history, collection, model="gpt-4.1-mini"):
    rewritten_query = rewrite_query(query, history, model)
    results = semantic_retrieve(collection, rewritten_query)
    best_context = rerank_results(results, rewritten_query, model)
    prompt = f"Answer the following question using the provided context. Include source information.\nQuestion: {query}\nContext: {best_context}\nHistory: {history}"
    response = openai_client.chat.completions.create(model=model, messages=[{"role": "user", "content": prompt}])
    return response.choices[0].message.content

---

## 7. Gradio Chat Interface Construction
Build a Gradio chat interface for interactive usage.

In [23]:
def build_gradio_chat(collection):
    with gr.Blocks() as demo:
        gr.Markdown("## Personal Knowledge Worker Chat")
        file_upload = gr.File(label="Upload .md or .txt file", file_types=["text"])
        chatbox = gr.Chatbot(type="messages", label="Conversation")
        user_input = gr.Textbox(label="Ask a question")
        submit_btn = gr.Button("Submit")
        history = []

        # Instead of storing collection in gr.State, store chunks
        chunks_state = gr.State([])

        def on_file_upload(file):
            if file is not None:
                with open(file.name, 'r', encoding='utf-8') as f:
                    doc_text = f.read()
                chunks = chunk_document(doc_text)
                return chunks
            return []

        def on_submit(query, chunks):
            collection = create_chromadb_collection(chunks)
            answer = conversational_qa(query, history, collection)
            history.append({'user': query, 'assistant': answer})
            return history

        file_upload.change(fn=on_file_upload, inputs=file_upload, outputs=chunks_state)
        submit_btn.click(fn=on_submit, inputs=[user_input, chunks_state], outputs=chatbox)
    return demo

---

## 8. Notebook Test Step
Quick sanity check for the workflow.

In [15]:
# Notebook Test Step

def test_workflow():
    docs = ingest_documents('knowledge_base/')
    chunks = []
    for doc in docs:
        chunks.extend(chunk_document(doc['text']))
    collection = create_chromadb_collection(chunks)
    answer = conversational_qa("What is the main topic?", [], collection)
    print("Sanity check answer:", answer)

In [24]:
def main():
    collection = create_chromadb_collection([])
    demo = build_gradio_chat(collection)
    demo.launch()

if __name__ == '__main__':
    main()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
